# 没有标签，怎样发现行为相近的客户群？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

            visits      basket  days_since  size
cluster                                         
0         2.163287   56.841850   40.310490    90
1        12.011051   46.108465    5.386059    90
2         3.979422  257.282038   18.123852    90


## 1. 补全：找到最大群
从 profiles 中找人数最多的群，核对输出人数。

In [2]:
largest = profiles['size'].idxmax()
assert int((customers['cluster']==largest).sum()) == int(profiles.loc[largest,'size'])
print(profiles.loc[largest])

visits         2.163287
basket        56.841850
days_since    40.310490
size          90.000000
Name: 0, dtype: float64


## 2. 修改：不标准化会怎样
对原单位特征运行相同 k，比较两种标签的调整兰德指数，不能直接逐个比较群编号。

In [3]:
from sklearn.metrics import adjusted_rand_score
unscaled = KMeans(n_clusters=selected_k,n_init=10,random_state=11).fit(customers[['visits','basket','days_since']])
agreement = adjusted_rand_score(model.labels_,unscaled.labels_)
assert -1 <= agreement <= 1
print('两种分群的一致性 ARI',agreement)

两种分群的一致性 ARI 1.0


## 3. 迁移：给新客户分组
用已拟合标准化器处理新客户，不能对一个新客户重新 fit。

In [4]:
new = pd.DataFrame({'visits':[3,11],'basket':[250,50],'days_since':[15,4]})
labels = model.predict(scaler.transform(new))
assert labels.shape == (2,) and set(labels).issubset(set(model.labels_))
print(new.assign(cluster=labels))

   visits  basket  days_since  cluster
0       3     250          15        2
1      11      50           4        1
